In [89]:
from dotenv import load_dotenv
import os

import pandas as pd
import numpy as np

import plotly.graph_objects as go

from data_functions import  get_fred_data
from util_functions import find_cols, remove_outliers, get_outliers

# Variables

In [90]:
start_date = '2000-01-01'

# Env

In [91]:
load_dotenv()

True

In [92]:
cotton_daily_filepath = os.getenv('COTTON_DAILY_DATA_LOCAL_FILEPATH')

In [93]:
cotton_daily_filepath

'/Users/zayankhan/Desktop/1212 Analytics/Cotton Data/MacroTrends/cotton-prices-historical-chart-data.csv'

# Load

In [94]:
cotton_df = pd.read_csv(cotton_daily_filepath, skiprows=15, index_col='date', parse_dates=['date']).rename(columns={' value': '$/lbs'})

In [95]:
cotton_df

,$/lbs
date,
1972-08-22,0.2673
1972-08-23,0.2703
1972-08-24,0.2706
1972-08-25,0.2722
1972-08-28,0.2704
...,...
2024-05-13,NaN
2024-05-14,NaN
2024-05-15,NaN


In [96]:
# adjust for cpi data

In [97]:
# futures price

In [98]:
# supply data 

In [99]:
# yield - by region 

In [100]:
# yield - total

In [101]:
# demand data

In [102]:
# weather data - growing regions - each column

In [103]:
# import/export data

# Transform

In [104]:
cotton_df['$/lbs'] = pd.to_numeric(cotton_df['$/lbs'])

In [105]:
cotton_df

,$/lbs
date,
1972-08-22,0.2673
1972-08-23,0.2703
1972-08-24,0.2706
1972-08-25,0.2722
1972-08-28,0.2704
...,...
2024-05-13,NaN
2024-05-14,NaN
2024-05-15,NaN


In [106]:
cotton_df = cotton_df[['$/lbs']].loc[start_date:].dropna()

In [107]:
cotton_df_outliers_rm = remove_outliers(cotton_df).ffill()

# View Price Data 

In [108]:
# Create the line plot with Plotly Graph Objects
fig = go.Figure(data=go.Scatter(x=cotton_df.index, y=cotton_df['$/lbs'], mode='lines'))

# Set graph layout, including dynamically set y-axis range
fig.update_layout(
    title='Cotton Prices ($/lbs)',
    xaxis_title='Date',
    yaxis_title='$/lbs',
    template='plotly_dark',
    xaxis=dict(
        rangeslider=dict(
            visible=True
        ),
        type="date"
    ),
    yaxis=dict(
        fixedrange=False  # Allow y-axis to be manually scalable by the user
    )
)

# Show the plot
fig.show()

# Calc Inflation Adjusted Cotton Px

In [109]:
ppi_codes_dict = {'finished_cotton': 'WPU034201', 'spun_cotton_yarns':'WPU032601', 'carded_spun_yarns': 'WPU03260113'}

In [110]:
cotton_df.index[-1]

Timestamp('2024-04-26 00:00:00')

In [111]:
ppi_values_cotton = get_fred_data(start_date, cotton_df.index[-1], ppi_codes_dict)
ppi_values_cotton['spun_cotton_yarns'] = ppi_values_cotton['spun_cotton_yarns'].ffill().bfill()

In [112]:
ppi_values_cotton

,finished_cotton,spun_cotton_yarns,carded_spun_yarns
DATE,,,
2000-01-01,114.700,92.300,NaN
2000-02-01,114.200,92.200,NaN
2000-03-01,113.100,92.500,NaN
2000-04-01,113.700,92.400,NaN
2000-05-01,113.000,92.400,NaN
...,...,...,...
2023-12-01,195.613,152.708,NaN
2024-01-01,196.064,152.708,NaN
2024-02-01,195.137,152.708,NaN


In [113]:
ppi_values_cotton['Cotton PPI'] = ppi_values_cotton.mean(axis=1)
#ToDo: weighted average with 50% weighting to spun yarn ppi

In [114]:
ppi_values_cotton.index.names = ['date']

In [115]:
ppi_values_cotton

,finished_cotton,spun_cotton_yarns,carded_spun_yarns,Cotton PPI
date,,,,
2000-01-01,114.700,92.300,NaN,103.5000
2000-02-01,114.200,92.200,NaN,103.2000
2000-03-01,113.100,92.500,NaN,102.8000
2000-04-01,113.700,92.400,NaN,103.0500
2000-05-01,113.000,92.400,NaN,102.7000
...,...,...,...,...
2023-12-01,195.613,152.708,NaN,174.1605
2024-01-01,196.064,152.708,NaN,174.3860
2024-02-01,195.137,152.708,NaN,173.9225


In [116]:
daily_ppi = pd.DataFrame(ppi_values_cotton['Cotton PPI'].resample('B').ffill()) #ffill bc you give rest of month the data 

In [117]:
cotton_df_outliers_rm = pd.merge(cotton_df_outliers_rm, daily_ppi, left_index=True, right_index=True, how='left').ffill() #no inflation yet for most recent data 

In [118]:
base_ppi_cotton = cotton_df_outliers_rm['Cotton PPI'].iloc[-1]

In [119]:
# so you do it with base ppi (todays price) vs a past date is bc the ppi index is a relational index so from x value it shows the % value the px on one date would be more or less
cotton_df_outliers_rm['$/lbs Inflation Adj'] = cotton_df_outliers_rm['$/lbs'] * (base_ppi_cotton / cotton_df_outliers_rm['Cotton PPI'])

In [120]:
cotton_df_outliers_rm

,$/lbs,Cotton PPI,$/lbs Inflation Adj
date,,,
2000-01-03,0.5107,103.500,0.868002
2000-01-04,0.5073,103.500,0.862224
2000-01-05,0.5156,103.500,0.876331
2000-01-06,0.5208,103.500,0.885169
2000-01-07,0.5396,103.500,0.917122
...,...,...,...
2024-04-22,0.8000,175.912,0.800000
2024-04-23,0.8000,175.912,0.800000
2024-04-24,0.8000,175.912,0.800000


In [123]:
# Create traces
cotton_px_trace= go.Scatter(
    x = cotton_df_outliers_rm.index,
    y = cotton_df_outliers_rm['$/lbs'],
    mode = 'lines',
    name = '$/lbs'
)

cotton_adj_px_trace = go.Scatter(
    x = cotton_df_outliers_rm.index,
    y = cotton_df_outliers_rm['$/lbs Inflation Adj'],
    mode = 'lines',
    name = '$/lbs Inflation Adj'
)

In [124]:
# Create the layout
layout = go.Layout(
    title = 'Cotton vs Inflation Adj ($/lbs)',
    xaxis = dict(title = 'Year-Month'),
    yaxis = dict(title = '($/lbs)')
)

In [125]:
# Create the figure
cotton_fig = go.Figure(data = [cotton_px_trace, cotton_adj_px_trace], layout = layout)

# Plot the figure
cotton_fig.show()